In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("workspace.bronze.customers")

df = (
    df
    .withColumn("signup_date", F.to_date("signup_date"))
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at", "dd-MM-yyyy HH:mm")
    )
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("first_name", F.trim("first_name"))
    .withColumn("last_name", F.trim("last_name"))
)



In [0]:
df.show()

In [0]:
window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("updated_at").desc())
)

silver_customers = (
    df
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(silver_customers)

In [0]:
bad_customers = silver_customers.filter(
    F.col("customer_id").isNull() |
    F.col("email").isNull()
)

display(bad_customers)

In [0]:
silver_customers = silver_customers.filter(
    F.col("customer_id").isNotNull() &
    F.col("email").isNotNull()
)

(
    silver_customers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.customers")
)

In [0]:
%sql
select *from workspace.silver.customers